In [3]:
from dotenv import load_dotenv
import os
import json
import re
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage
from pathlib import Path
import sqlite3

load_dotenv(override=True)
api_key = os.environ["OPENAI_API_KEY"]



In [ ]:
# DB_PATH = 'database/ESGdata.db'

# def run_ai_reasoning_and_update():
#     # 1. AI 모델 설정 (GPT-5 기반)
#     llm = ChatOpenAI(
#     model = 'gpt-5-nano-2025-08-07',
#     temperature = 0.1)
    
#     conn = sqlite3.connect(DB_PATH)
#     cursor = conn.cursor()

#     # 2. 진단이 필요한 이상치 데이터 추출 (에너지 + 생산량 조인)
#     query = """
#     SELECT o.id, s.site_id, s.reporting_date, s.metric, s.value, 
#            o.layer, o.threshold, a.production_qty
#     FROM outlier_results o
#     JOIN standard_usage s ON o.std_id = s.id
#     JOIN activity_data a ON s.site_id = a.site_id AND s.reporting_date = a.reporting_date
#     WHERE o.analysis_summary IS NULL -- 아직 진단되지 않은 건만 추출
#     """
#     outliers = cursor.execute(query).fetchall()

#     if not outliers:
#         print("💡 진단할 새로운 이상치 데이터가 없습니다.")
#         conn.close()
#         return

#     for o_id, site, date, metric, val, layer, limit, prod in outliers:
#         # 3. 분석가 페르소나를 주입한 프롬프트 구성
#         system_msg = SystemMessage(content="귀하는 ESG 데이터 분석 전문가입니다. 제공된 수치와 알고리즘 결과를 바탕으로 이상치의 원인을 진단하십시오.")
        
#         prompt = f"""
#         [이상치 탐지 팩트]
#         - 사업장/지표: {site} / {metric}
#         - 발생 시점: {date}
#         - 탐지 수치: {val} (임계치: {limit})
#         - 위반 계층: {layer}
#         - 당시 생산량: {prod} Ton
        
#         [진단 규칙]
#         1. Layer 2 위반이며 수치가 임계치를 1.5배 이상 상회 시 '단위 오기입(kWh->MWh)' 가능성을 최우선으로 제시.
#         2. 생산량 변동폭보다 에너지 사용량 변동폭이 현저히 크면 '운영 비효율' 또는 '계량기 오류' 진단.
#         3. 위 내용을 바탕으로 담당자가 즉시 확인해야 할 사항을 3줄 내외로 정리하십시오.
#         """

#         # 4. LLM 추론 실행
#         response = llm.invoke([system_msg, HumanMessage(content=prompt)])
#         analysis_result = response.content

#         # 5. DB에 진단 결과 업데이트
#         cursor.execute("UPDATE outlier_results SET analysis_summary = ? WHERE id = ?", (analysis_result, o_id))
#         print(f"✅ AI 진단 완료: {site} ({date})")

#     conn.commit()
#     conn.close()
#     print("🚀 모든 이상치에 대한 AI 분석 보고서가 DB에 저장되었습니다.")

# if __name__ == "__main__":
#     run_ai_reasoning_and_update()

✅ AI 진단 완료: Site A (2025-07)
✅ AI 진단 완료: Site A (2025-10)
✅ AI 진단 완료: Site B (2025-11)
🚀 모든 이상치에 대한 AI 분석 보고서가 DB에 저장되었습니다.


In [ ]:
# def run_advanced_ai_reasoning1():
#     # 1. AI 모델 설정
#     llm = ChatOpenAI(model='gpt-4o', temperature=0.1)
    
#     # 2. 시스템 인스트럭션 (팀장님 수정안 반영)
#     system_instruction = """
#     너는 ESG 데이터의 정합성을 최종 진단하고 현장에 명확한 가이드를 내리는 'ESG 실무 데이터 감사인'이다. 
#     딱딱한 시스템 용어(L1, L2, L3)를 지우고, 데이터의 성격에 맞는 '현장용 비즈니스 언어'로 재구성하라.

#     [실무 용어 치환 매핑]
#     - L1 (Z-score/YoY) -> '과거 운영 기록 대비 변동'
#     - L2 (Physical Limit) -> '설비 가동 한계 초과'
#     - L3 (Intensity Deviation) -> '에너지 투입 효율 저하'

#     [진단 및 해설 원칙]
#     1. 사람 중심의 해설: 'L1 PASS' 같은 표현 대신 서술형으로 풀어서 설명할 것.
#     2. 직관적 수치 배수: '평소 대비 X배', '상한선 대비 Y% 초과'를 사용하여 심각성을 부각할 것.
#     3. Extreme Rule 적용: 수치가 임계치의 5배(500%)를 초과할 경우, 운영 이슈가 아닌 '단위 오기입(kWh ↔ MWh)'으로 단정하여 안내할 것.
    
#     [상황별 체크리스트 가이드]
#     - 설비 가동 한계 초과 시: 데이터 단위(kWh/MWh) 오기입 대조, 계측기 고장 여부, 자릿수 오타 확인.
#     - 에너지 투입 효율 저하 시: 생산 실적 누락 여부 대조, 공정 내 에너지 누수, 설비 공회전 시간 확인.
#     """

#     conn = sqlite3.connect(DB_PATH)
#     cursor = conn.cursor()

#     # 진단 대상을 가져옵니다 (outlier_results + standard_usage + activity_data 조인)
#     query = """
#     SELECT o.id, s.site_id, s.reporting_date, s.metric, s.value, 
#            o.layer, o.threshold, a.production_qty, s.id as std_id
#     FROM outlier_results o
#     JOIN standard_usage s ON o.std_id = s.id
#     JOIN activity_data a ON s.site_id = a.site_id AND s.reporting_date = a.reporting_date
#     WHERE o.analysis_summary IS NULL
#     """
#     targets = cursor.execute(query).fetchall()

#     if not targets:
#         print("💡 진단할 새로운 데이터가 없습니다.")
#         conn.close()
#         return

#     for o_id, site, date, metric, val, layer, limit, prod, std_id in targets:
#         # LLM에게 전달할 구체적 팩트 구성
#         user_content = f"""
#         진단 대상 데이터:
#         - 사업장: {site} ({date})
#         - 지표: {metric}
#         - 현재 측정값: {val}
#         - 시스템 임계치: {limit}
#         - 탐지된 이상 징후: {layer}
#         - 해당 월 생산량: {prod} Ton
        
#         위 정보를 바탕으로 [출력 JSON 스펙]에 맞춰 보고서를 작성하라. 
#         출력은 반드시 순수 JSON 형태여야 한다.
#         """

#         response = llm.invoke([
#             SystemMessage(content=system_instruction),
#             HumanMessage(content=user_content)
#         ])
        
#         # 결과 저장 (JSON 문자열 그대로 저장하거나 파싱 후 특정 필드만 저장)
#         analysis_json = response.content
#         cursor.execute("UPDATE outlier_results SET analysis_summary = ? WHERE id = ?", (analysis_json, o_id))
#         print(f"✅ {site} {date} 진단 완료")

#     conn.commit()
#     conn.close()

# if __name__ == "__main__":
#     run_advanced_ai_reasoning1()

✅ Site A 2025-07 진단 완료
✅ Site A 2025-10 진단 완료
✅ Site B 2025-11 진단 완료


In [17]:
def run_advanced_ai_reasoning2():
    # 1. AI 모델 설정
    llm = ChatOpenAI(model='gpt-4o', temperature=0.1)
    
    # 2. 시스템 인스트럭션 (팀장님 수정안 반영)
    system_instruction = """
    너는 ESG 데이터의 정합성을 최종 진단하고 현장에 명확한 가이드를 내리는 'ESG 실무 데이터 감사인'이다. 
    딱딱한 시스템 용어(L1, L2, L3)를 지우고, 데이터의 성격에 맞는 '현장용 비즈니스 언어'로 재구성하라.

    [실무 용어 치환 매핑]
    - L1 (Z-score/YoY) -> '과거 운영 기록 대비 변동'
    - L2 (Physical Limit) -> '설비 가동 한계 초과'
    - L3 (Intensity Deviation) -> '에너지 투입 효율 저하'

    [진단 및 해설 원칙]
    1. 사람 중심의 해설: 'L1 PASS' 같은 표현 대신 서술형으로 풀어서 설명할 것.
    2. 직관적 수치 배수: '평소 대비 X배', '상한선 대비 Y% 초과'를 사용하여 심각성을 부각할 것.
    3. Extreme Rule 적용: 수치가 임계치의 5배(500%)를 초과할 경우, 운영 이슈가 아닌 '단위 오기입(kWh ↔ MWh)'으로 단정하여 안내할 것.
    
    [상황별 체크리스트 가이드]
    - 설비 가동 한계 초과 시: 데이터 단위(kWh/MWh) 오기입 대조, 계측기 고장 여부, 자릿수 오타 확인.
    - 에너지 투입 효율 저하 시: 생산 실적 누락 여부 대조, 공정 내 에너지 누수, 설비 공회전 시간 확인.
    """

    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    # 진단 대상을 가져옵니다 (outlier_results + standard_usage + activity_data 조인)
    query = """
    SELECT o.id, s.site_id, s.reporting_date, s.metric, s.value, 
            o.layer, o.threshold, a.production_qty
    FROM outlier_results o
    JOIN standard_usage s ON o.std_id = s.id
    JOIN activity_data a ON s.site_id = a.site_id AND s.reporting_date = a.reporting_date
    WHERE s.v_status = 2 AND o.analysis_summary IS NULL
    """
    targets = cursor.execute(query).fetchall()

    if not targets:
        print("💡 진단할 새로운 데이터가 없습니다.")
        conn.close()
        return

    for o_id, site, date, metric, val, layer, limit, prod in targets:
        # LLM에게 전달할 구체적 팩트 구성
        # f-string 내에서 중괄호 사용을 위해 {{ }} 형식을 사용합니다.
        user_content = f"""
        [진단 대상 데이터]
        - 사업장: {site} ({date})
        - 지표: {metric}
        - 측정값: {val}
        - 시스템 임계치: {limit}
        - 탐지 계층: {layer}
        - 해당 월 생산량: {prod} Ton
        
        위 정보를 바탕으로 아래 JSON 형식에 맞춰 보고서를 작성하라.
        {{
            "outliers": [
            {{
                "이상치_식별자": "{site}_{date}_{metric}",
                "위험_등급": "Critical/Major/Warning 중 택1",
                "진단_요약": "현장 담당자용 핵심 메시지",
                "판단_근거_및_해설": "비즈니스 언어로 풀어서 설명",
                "추론_가설": "데이터 오기입 또는 현장 이슈 추정",
                "현장_체크리스트": ["점검항목1", "점검항목2", "점검항목3"]
            }}
            ]
        }}
        """

        response = llm.invoke([
            SystemMessage(content=system_instruction),
            HumanMessage(content=user_content)
        ])
        
        # AI 응답에서 마크다운 태그(```json)가 포함될 경우 제거
        analysis_json = response.content.replace("```json", "").replace("```", "").strip()
        cursor.execute("UPDATE outlier_results SET analysis_summary = ? WHERE id = ?", (analysis_json, o_id))
        print(f"✅ {site} {date} 진단 완료")

    conn.commit()
    conn.close()

if __name__ == "__main__":
    run_advanced_ai_reasoning2()

✅ Site A 2025-07 진단 완료
✅ Site A 2025-10 진단 완료
✅ Site B 2025-11 진단 완료
